In [1]:
import numpy as np
import pandas as pd

In [2]:
transaction_matches = pd.read_csv("../temp/transaction_transaction_matches.csv")
display(transaction_matches.head(5))
display(transaction_matches.columns)
display(transaction_matches.shape)

,t_DATE,t_PERIOD,t_AUDITYPE,t_STORECODE,t_DLRCODE,t_ITEMCODE,t_NEW_CODES,t_CATEGORY,t_MANUFACTURE,t_BRAND,...,m_mbrand,m_brand,m_sku,m_packtype,m_base_pack,m_flavor,m_color,m_wght,m_uom,m_mrp
0,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765334,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT,CAMEL CONNECT ORIGINAL,HL 20'S,HL,20,FULL FLAVOUR,NaN,20.0,NO,161.5
1,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765334,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT,CAMEL CONNECT MINT CRUSH,HL 20'S,HL,20,FRESH,NaN,20.0,NO,239.0
2,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765334,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL,CAMEL BLUE,HL 20'S,HL,20,LIGHT,NaN,20.0,NO,300.0
3,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765335,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT,CAMEL CONNECT MINT CRUSH,HL 20'S,HL,20,FRESH,NaN,20.0,NO,239.0
4,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765335,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT,CAMEL CONNECT ORIGINAL,HL 20'S,HL,20,FULL FLAVOUR,NaN,20.0,NO,161.5


Index(['t_DATE', 't_PERIOD', 't_AUDITYPE', 't_STORECODE', 't_DLRCODE',
       't_ITEMCODE', 't_NEW_CODES', 't_CATEGORY', 't_MANUFACTURE', 't_BRAND',
       't_ITEMDESC', 't_MRP', 't_PACKSIZE', 't_PACKTYPE', 't_COMMENTS',
       't_IMAGE', 't_CODE COMMENT', 't_FLAG', 'rank', 'matched_itemcode',
       'distance', 'm_catcode', 'm_category', 'm_subcat', 'm_ssubcat',
       'm_company', 'm_mbrand', 'm_brand', 'm_sku', 'm_packtype',
       'm_base_pack', 'm_flavor', 'm_color', 'm_wght', 'm_uom', 'm_mrp'],
      dtype='object')

(14208, 36)

In [3]:
transaction_matches["transaction_id"] = transaction_matches.index // 3
display(transaction_matches.head(5))
display(transaction_matches.columns)
display(transaction_matches.shape)

,t_DATE,t_PERIOD,t_AUDITYPE,t_STORECODE,t_DLRCODE,t_ITEMCODE,t_NEW_CODES,t_CATEGORY,t_MANUFACTURE,t_BRAND,...,m_brand,m_sku,m_packtype,m_base_pack,m_flavor,m_color,m_wght,m_uom,m_mrp,transaction_id
0,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765334,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT ORIGINAL,HL 20'S,HL,20,FULL FLAVOUR,NaN,20.0,NO,161.5,0
1,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765334,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT MINT CRUSH,HL 20'S,HL,20,FRESH,NaN,20.0,NO,239.0,0
2,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765334,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL BLUE,HL 20'S,HL,20,LIGHT,NaN,20.0,NO,300.0,0
3,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765335,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT MINT CRUSH,HL 20'S,HL,20,FRESH,NaN,20.0,NO,239.0,1
4,11/12/2024,202412.0,3.0,2.110100e+09,2.110103e+10,1.733828e+12,99765335,105.0,JT INTERNATIONAL - SA,CAMEL CONNECT,...,CAMEL CONNECT ORIGINAL,HL 20'S,HL,20,FULL FLAVOUR,NaN,20.0,NO,161.5,1


Index(['t_DATE', 't_PERIOD', 't_AUDITYPE', 't_STORECODE', 't_DLRCODE',
       't_ITEMCODE', 't_NEW_CODES', 't_CATEGORY', 't_MANUFACTURE', 't_BRAND',
       't_ITEMDESC', 't_MRP', 't_PACKSIZE', 't_PACKTYPE', 't_COMMENTS',
       't_IMAGE', 't_CODE COMMENT', 't_FLAG', 'rank', 'matched_itemcode',
       'distance', 'm_catcode', 'm_category', 'm_subcat', 'm_ssubcat',
       'm_company', 'm_mbrand', 'm_brand', 'm_sku', 'm_packtype',
       'm_base_pack', 'm_flavor', 'm_color', 'm_wght', 'm_uom', 'm_mrp',
       'transaction_id'],
      dtype='object')

(14208, 37)

In [4]:
# Pivot the DataFrame so that each rank becomes a separate column
df_pivot = transaction_matches.pivot_table(
    index="transaction_id",
    columns="rank",
    values="matched_itemcode",
    aggfunc="first"
).reset_index()

# Rename columns for clarity
df_pivot = df_pivot.rename(columns={
    1: "rank1_matched_itemcode",
    2: "rank2_matched_itemcode",
    3: "rank3_matched_itemcode"
})

df_truth = transaction_matches[["transaction_id", "t_NEW_CODES"]].drop_duplicates()

df = df_pivot.merge(df_truth, on="transaction_id", how="left")

# Keep only required columns
df = df[[
    "t_NEW_CODES",
    "rank1_matched_itemcode",
    "rank2_matched_itemcode",
    "rank3_matched_itemcode"
]]

display(df.head(5))
display(df.columns)
display(df.shape)

,t_NEW_CODES,rank1_matched_itemcode,rank2_matched_itemcode,rank3_matched_itemcode
0,99765334,99765334,99765335,90110150
1,99765335,99765335,99765334,90110150
2,20121653,20121653,99763614,20121098
3,52357,52357,20121098,99764266
4,99765336,99765336,99765473,99765207


Index(['t_NEW_CODES', 'rank1_matched_itemcode', 'rank2_matched_itemcode',
       'rank3_matched_itemcode'],
      dtype='object')

(4736, 4)

# Metrics Documentation

Given a DataFrame with the following columns:

* `t_NEW_CODES` → the true/actual item code.
* `rank1_matched_itemcode` → predicted code at rank 1.
* `rank2_matched_itemcode` → predicted code at rank 2.
* `rank3_matched_itemcode` → predicted code at rank 3.

The goal is to evaluate the model using **rank-based accuracies** and **classification-based metrics**.

---

## 1. Rank Accuracies

For each row, compare the true code against the predicted codes at each rank.

* **Rank 1 Accuracy**:
  Fraction of rows where `t_NEW_CODES == rank1_matched_itemcode`.

  $$
  \text{Rank1 Accuracy} = \frac{\text{Number of matches at rank1}}{\text{Total rows}}
  $$

* **Rank 2 Accuracy**:
  Fraction of rows where `t_NEW_CODES == rank2_matched_itemcode`.

  $$
  \text{Rank2 Accuracy} = \frac{\text{Number of matches at rank2}}{\text{Total rows}}
  $$

* **Rank 3 Accuracy**:
  Fraction of rows where `t_NEW_CODES == rank3_matched_itemcode`.

  $$
  \text{Rank3 Accuracy} = \frac{\text{Number of matches at rank3}}{\text{Total rows}}
  $$

---

## 2. Presence-based Evaluation

Instead of focusing on exact rank, we check whether the true code appears in the top-3 predictions.

* **True Positive (TP):**
  Rows where `t_NEW_CODES` appears in `[rank1, rank2, rank3]`.

* **False Negative (FN):**
  Rows where `t_NEW_CODES` does not appear in `[rank1, rank2, rank3]`.

* **False Positive (FP):**
  Since the model always outputs predictions, if none of the predictions match the true code, we count this as a false positive. In this setup, `FP = FN`.

* **True Negative (TN):**
  There are no true negatives in this dataset because every row has a valid `t_NEW_CODES`. Therefore, `TN = 0`.

---

## 3. Metrics Definitions

Using the above:

* **Precision**
  Out of all rows predicted as containing the true code, how many were correct.

  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$

* **Recall (Sensitivity)**
  Out of all rows with a true code, how many were detected.

  $$
  \text{Recall} = \frac{TP}{TP + FN}
  $$

* **Specificity**
  Fraction of negatives correctly identified. Since there are no true negatives, specificity will always be 0 in this setup.

  $$
  \text{Specificity} = \frac{TN}{TN + FP}
  $$

* **Type I Error (False Positive Rate)**
  Fraction of negatives incorrectly predicted as positives. With no true negatives, this collapses to 1 whenever FP > 0.

  $$
  \text{Type I Error} = \frac{FP}{FP + TN}
  $$

* **Type II Error (False Negative Rate)**
  Fraction of positives missed by the model.

  $$
  \text{Type II Error} = \frac{FN}{FN + TP}
  $$

---

## 4. Notes

* Rank accuracies are straightforward and reliable.
* Precision and recall are meaningful for presence detection.
* Specificity and Type I error are not meaningful in this setup due to the absence of true negatives. To make them meaningful, negatives would need to be defined at the item level (e.g., every non-true prediction treated individually as a candidate negative).

In [5]:
# Convert t_NEW_CODES to a numeric type, handling potential non-numeric values and NaNs
df["t_NEW_CODES"] = pd.to_numeric(df["t_NEW_CODES"], errors='coerce').astype('Int64')

# Rank accuracies
rank1_acc = (df["t_NEW_CODES"] == df["rank1_matched_itemcode"]).mean()
rank2_acc = (df["t_NEW_CODES"] == df["rank2_matched_itemcode"]).mean()
rank3_acc = (df["t_NEW_CODES"] == df["rank3_matched_itemcode"]).mean()

# Check if true code is present in any of the predictions
df["present_in_any"] = df.apply(
    lambda row: row["t_NEW_CODES"] in [
        row["rank1_matched_itemcode"],
        row["rank2_matched_itemcode"],
        row["rank3_matched_itemcode"]
    ] if pd.notna(row["t_NEW_CODES"]) else False, axis=1
)

TP = df["present_in_any"].sum()
FN = (~df["present_in_any"]).sum()

# In this closed setup: FP = rows predicted but wrong = FN, TN = 0
FP = FN
TN = 0

print("TP: ", TP)
print("FP: ", FP)
print("TN: ", TN)
print("FN: ", FN)
print("Total: ", TP+FP+TN+FN)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
type1_error = FP / (FP + TN) if (FP + TN) > 0 else 0
type2_error = FN / (FN + TP) if (FN + TP) > 0 else 0

# Final metrics table
metrics = pd.DataFrame({
    "rank1_acc": [rank1_acc],
    "rank2_acc": [rank2_acc],
    "rank3_acc": [rank3_acc],
    "precision": [precision],
    "recall": [recall],
    "specificity": [specificity],
    "type1_error": [type1_error],
    "type2_error": [type2_error],
})

display(metrics)

TP:  4243
FP:  493
TN:  0
FN:  493
Total:  5229


,rank1_acc,rank2_acc,rank3_acc,precision,recall,specificity,type1_error,type2_error
0,0.819123,0.07733,0.038334,0.895904,0.895904,0.0,1.0,0.104096
